Link Dataset & Config

In [1]:
import tensorflow as tf
from object_detection.utils import config_util
from object_detection.protos import pipeline_pb2
from google.protobuf import text_format
import os

# ==============================================================================
# ⚙️ CONFIGURATION (Auto-filled based on your screenshots)
# ==============================================================================
# 1. Pipeline Config Path
pipeline_config = r'D:\Documents\University\Adv Proj\Model\tf_workspace\pre_trained_models\ssd_mobilenet_v2\pipeline.config'

# 2. Pre-trained Checkpoint (Must point to the file prefix 'ckpt-0')
checkpoint_path = r'D:\Documents\University\Adv Proj\Model\tf_workspace\pre_trained_models\ssd_mobilenet_v2\checkpoint\ckpt-0'

# 3. Label Map (Inside your train folder)
label_map_path = r'D:\Documents\University\Adv Proj\Dataset\tfrecord\CV-Driver-Distraction-3\train\objects_label_map.pbtxt'

# 4. TF Records (Roboflow structure: Train -> Train, Valid -> Test)
train_record_path = r'D:\Documents\University\Adv Proj\Dataset\tfrecord\CV-Driver-Distraction-3\train\objects.tfrecord'
test_record_path = r'D:\Documents\University\Adv Proj\Dataset\tfrecord\CV-Driver-Distraction-3\valid\objects.tfrecord'

# 5. Model Settings
num_classes = 3       # Phone, Safe, Texting
batch_size = 4        # Safe for RTX 3060 Ti

# ==============================================================================
# 🛠️ EXECUTE LINKING
# ==============================================================================
print(f"Linking config for SSD 640x640...")

cwd = os.getcwd()
pipeline_config_path = os.path.join(cwd, pipeline_config)
configs = config_util.get_configs_from_pipeline_file(pipeline_config_path)

# Load config objects
model_config = configs['model']
train_config = configs['train_config']
input_config = configs['train_input_config']
eval_config = configs['eval_input_config']

# 1. Set Classes
model_config.ssd.num_classes = num_classes

# 2. Set Checkpoint & Batch Size
train_config.batch_size = batch_size
train_config.fine_tune_checkpoint = os.path.join(cwd, checkpoint_path)
train_config.fine_tune_checkpoint_type = "detection"
train_config.use_bfloat16 = False 

# 3. Link TFRecords & Label Map
input_config.label_map_path = os.path.join(cwd, label_map_path)
input_config.tf_record_input_reader.input_path[:] = [os.path.join(cwd, train_record_path)]

eval_config.label_map_path = os.path.join(cwd, label_map_path)
eval_config.tf_record_input_reader.input_path[:] = [os.path.join(cwd, test_record_path)]

# 4. Save Config
pipeline_proto = config_util.create_pipeline_proto_from_configs(configs)
config_util.save_pipeline_config(pipeline_proto, os.path.dirname(pipeline_config_path))

print("✅ SUCCESS: Dataset Linked!")
print(f"   - Checkpoint: {train_config.fine_tune_checkpoint}")
print(f"   - Classes: {model_config.ssd.num_classes}")

d:\Terminal\ancondaEnviroment\tf_od_fix\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
d:\Terminal\ancondaEnviroment\tf_od_fix\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\Terminal\ancondaEnviroment\tf_od_fix\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-a

Linking config for SSD 640x640...
INFO:tensorflow:Writing pipeline config file to D:\Documents\University\Adv Proj\Model\tf_workspace\pre_trained_models\ssd_mobilenet_v2\pipeline.config
✅ SUCCESS: Dataset Linked!
   - Checkpoint: D:\Documents\University\Adv Proj\Model\tf_workspace\pre_trained_models\ssd_mobilenet_v2\checkpoint\ckpt-0
   - Classes: 3


Training Here (Engine)

In [3]:
import os
import sys
import subprocess

# ==========================================
# 🚀 START TRAINING ENGINE
# ==========================================
# Paths
pipeline_config = os.path.join(os.getcwd(), 'pre_trained_models', 'ssd_mobilenet_v2', 'pipeline.config')
model_dir = r'D:\Documents\University\Adv Proj\Train Model\SSD'
training_script = os.path.join(os.getcwd(), 'models', 'research', 'object_detection', 'model_main_tf2.py')

# Command (Uses the current environment's Python)
command = [
    sys.executable,
    training_script,
    f"--pipeline_config_path={pipeline_config}",
    f"--model_dir={model_dir}",
    "--alsologtostderr",
    "--num_train_steps=5000",      # Stops after 5000 steps
    "--sample_1_of_n_eval_examples=1"
    "--use_tpu=False"
]
# Set environment variables to prevent XLA issues
env = os.environ.copy()
env["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
env["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=none"

print("----------------------------------------------------")
print(f"📜 Config: {pipeline_config}")
print(" STARTING TRAINING... (Please wait ~60s for CUDA to load)")
print("----------------------------------------------------")

# Run and stream output
process = subprocess.Popen(command,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT,
                            text=True,
                            bufsize=1,
                            env=env
                            )

try:
    for line in process.stdout:
        print(line, end="") # 🟢 Look for 'loss=' here!
except KeyboardInterrupt:
    print("\n🛑 Training stopped by user.")
    process.terminate()

process.wait()

----------------------------------------------------
📜 Config: d:\Documents\University\Adv Proj\Model\tf_workspace\pre_trained_models\ssd_mobilenet_v2\pipeline.config
 STARTING TRAINING... (Please wait ~60s for CUDA to load)
----------------------------------------------------
d:\Terminal\ancondaEnviroment\tf_od_fix\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.25). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
d:\Terminal\ancondaEnviroment\tf_od_fix\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python versi

1